In [ ]:
import torchaudio as ta
from chatterbox_infer.mtl_tts import ChatterboxMultilingualTTS
import time
import torch

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")

In [ ]:
from IPython.display import Audio
import time
import torch

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Listen carefully: this is the most important part.",
]

total_seconds = 0
total_latency = 0
for text in texts:
    st = time.time()

    wav = model.generate(
        text, 
        audio_prompt_path='./elevenlabs3.mp3', 
        language_id='en',
        exaggeration=0.4,
        cfg_weight=0.5,
        temperature=0.7,
        repetition_penalty=1.3,
        min_p=0.02,
        top_p=0.9
    )
    total_seconds += wav.shape[-1]/24000
    total_latency += time.time() - st
    # print(text)
    # print(f"[For {wav.shape[-1]/24000}s] taken: {time.time() - st:.3f}")
    display(Audio(wav, rate=24000))

print(f"1초 생성에 {total_latency/total_seconds:.3f}초 걸림")

In [ ]:
import torch
from IPython.display import Audio
import time

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Come on, hurry up! We’re going to be late!",
    "Oh please, don’t tell me you forgot again.",
    "Wait, what?! That makes no sense at all.",
    "I told you already—I’m not going back there.",
    "Ha! That was the funniest thing I’ve ever seen.",
    "Ugh, I’m so tired… I just need one more coffee.",
    "Look out! That car is coming way too fast!",
    "Yes, yes, yes! Finally, it worked!",
    "Oh no… I really messed up this time.",
    "Are you kidding me right now?",
    "Listen carefully: this is the most important part.",
]

total_seconds = 0
total_latency = 0

for text in texts:
    st = time.time()
    wavs = []
    last_length = 0
    async for evt in model.generate_stream(
        text,
        audio_prompt_path='./elevenlabs3.mp3',
        language_id='en',
        exaggeration=0.4,
        cfg_weight=0.5,
        temperature=0.7,
        repetition_penalty=1.3,
        min_p=0.02,
        top_p=0.9
    ):
        if evt["type"] == "chunk":
            wav = evt["audio"]
            wavs.append(wav[:, max(last_length, 0):])
            last_length = wav.shape[-1]
        elif evt["type"] == "eos":
            full = torch.cat(wavs, dim=-1).cpu().numpy()
            # print(f"✅ 스트리밍 끝! {text} ({full.shape[-1]/24000:.2f}s, {time.time()-st:.2f}s 소요)")
            total_seconds += full.shape[-1]/24000
            total_latency += time.time() - st
            display(Audio(full, rate=24000))

print(f"1초 생성에 {total_latency/total_seconds:.3f}초 걸림")

In [ ]:
def t3_to(model, dtype):
    model.t3.to(dtype=dtype)
    model.conds.t3.to(dtype=dtype)
    torch.cuda.empty_cache()
    return model

t3_to(model, torch.bfloat16)

In [ ]:
import torchaudio as ta
from chatterbox_infer.mtl_tts_stream import ChatterboxMultilingualTTS
import time

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")

In [ ]:
import torch
from IPython.display import Audio
import time

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Come on, hurry up! We’re going to be late!",
    "Oh please, don’t tell me you forgot again.",
    "Wait, what?! That makes no sense at all.",
    "I told you already—I’m not going back there.",
    "Ha! That was the funniest thing I’ve ever seen.",
    "Ugh, I’m so tired… I just need one more coffee.",
    "Look out! That car is coming way too fast!",
    "Yes, yes, yes! Finally, it worked!",
    "Oh no… I really messed up this time.",
    "Are you kidding me right now?",
    "Listen carefully: this is the most important part.",
]

total_seconds = 0
total_latency = 0

for text in texts:
    st = time.time()
    wavs = []
    last_length = 0
    async for evt in model.generate_stream(
        text,
        audio_prompt_path='./elevenlabs3.mp3',
        language_id='en',
        exaggeration=0.6,
        cfg_weight=0.4,
        temperature=0.7,
        repetition_penalty=1.3,
        min_p=0.02,
        top_p=0.9,
    ):
        if evt["type"] == "chunk":
            wav = evt["audio"]
            wavs.append(wav[:, max(last_length, 0):])
            last_length = wav.shape[-1]
        elif evt["type"] == "eos":
            full = torch.cat(wavs, dim=-1).cpu().numpy()
            # print(f"✅ 스트리밍 끝! {text} ({full.shape[-1]/24000:.2f}s, {time.time()-st:.2f}s 소요)")
            # display(Audio(full, rate=24000))
            total_seconds += full.shape[-1]/24000
            total_latency += time.time() - st

print(f"1초 생성에 {total_latency/total_seconds:.3f}초 걸림")